In [2]:
import os

print("cwd =", os.getcwd())
print("Existe dossier Mac ? ", os.path.isdir("/Users/cefim/Downloads/datalake_data"))
print("Existe annonces ? ", os.path.exists("/Users/cefim/Downloads/datalake_data/annonces_immo.json"))

cwd = /Users/cefim
Existe dossier Mac ?  True
Existe annonces ?  True


In [3]:
from minio import Minio

client = Minio(
    "localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)

print("Buckets:", [b.name for b in client.list_buckets()])

Buckets: ['immobilier-curated', 'immobilier-raw', 'immobilier-refined']


In [4]:
import os

BASE_DIR = "/Users/cefim/Downloads/datalake_data"
print("Fichiers dans BASE_DIR:", os.listdir(BASE_DIR))

for f in ["annonces_immo.json", "images_metadata.json", "donnees_geo.geojson", "donnees_geo.csv"]:
    print(f, "=>", os.path.exists(os.path.join(BASE_DIR, f)))

Fichiers dans BASE_DIR: ['.DS_Store', 'donnees_geo.geojson', 'annonces_immo.json', 'images_metadata.json']
annonces_immo.json => True
images_metadata.json => True
donnees_geo.geojson => True
donnees_geo.csv => False


In [5]:
import os
from minio import Minio
from minio.error import S3Error

BASE_DIR = "/Users/cefim/Downloads/datalake_data"
BUCKET = "immobilier-raw"

client = Minio(
    "localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)

# Adapte ici selon les fichiers qui existent vraiment
fichiers = [
    ("annonces_immo.json", "annonces/"),
    ("images_metadata.json", "images/"),
]

# Geo: on choisit le bon format
if os.path.exists(os.path.join(BASE_DIR, "donnees_geo.geojson")):
    fichiers.append(("donnees_geo.geojson", "geographiques/"))
elif os.path.exists(os.path.join(BASE_DIR, "donnees_geo.csv")):
    fichiers.append(("donnees_geo.csv", "geographiques/"))
else:
    print("⚠ Aucun fichier geo trouvé (ni donnees_geo.geojson ni donnees_geo.csv)")

for nom_fichier, prefix in fichiers:
    local_path = os.path.join(BASE_DIR, nom_fichier)
    object_name = prefix + nom_fichier

    if not os.path.exists(local_path):
        print("❌ Introuvable:", local_path)
        continue

    try:
        client.fput_object(BUCKET, object_name, local_path)
        print("✅ Upload:", local_path, "->", f"{BUCKET}/{object_name}")
    except S3Error as e:
        print("❌ Erreur upload:", nom_fichier, e)

✅ Upload: /Users/cefim/Downloads/datalake_data/annonces_immo.json -> immobilier-raw/annonces/annonces_immo.json
✅ Upload: /Users/cefim/Downloads/datalake_data/images_metadata.json -> immobilier-raw/images/images_metadata.json
✅ Upload: /Users/cefim/Downloads/datalake_data/donnees_geo.geojson -> immobilier-raw/geographiques/donnees_geo.geojson


In [6]:
from minio import Minio

client = Minio("localhost:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)

print("Buckets:", [b.name for b in client.list_buckets()])
for o in client.list_objects("immobilier-raw", recursive=True):
    print(o.object_name, o.size)

Buckets: ['immobilier-curated', 'immobilier-raw', 'immobilier-refined']
annonces/annonces_immo.json 32740
curated/catalogues/catalog_biens.csv 9085
curated/catalogues/catalog_files.csv 886
curated/catalogues/catalog_images_ml.csv 10440
geographiques/donnees_geo.geojson 20565
images/images_metadata.json 66385


In [7]:
#Préparer un dossier “pipeline” local (pour que ce soit Airflow-ready ensuite)

In [8]:
import os
from pathlib import Path

ROOT = Path.home() / "immobilier_pipeline"
(STAGING := ROOT / "staging").mkdir(parents=True, exist_ok=True)
(OUTPUTS := ROOT / "outputs").mkdir(parents=True, exist_ok=True)
(MODELS := ROOT / "models").mkdir(parents=True, exist_ok=True)

print("ROOT =", ROOT)

ROOT = /Users/cefim/immobilier_pipeline


In [9]:
#Installer les libs ML

In [10]:
pip install pandas numpy scikit-learn joblib minio

Note: you may need to restart the kernel to use updated packages.


In [11]:
objs = [o.object_name for o in client.list_objects(BUCKET, recursive=True)]
print("Objets MinIO:")
for x in objs:
    print(" -", x)

Objets MinIO:
 - annonces/annonces_immo.json
 - curated/catalogues/catalog_biens.csv
 - curated/catalogues/catalog_files.csv
 - curated/catalogues/catalog_images_ml.csv
 - geographiques/donnees_geo.geojson
 - images/images_metadata.json


In [12]:
from pathlib import Path

ROOT = Path.home() / "immobilier_pipeline"
STAGING = ROOT / "staging"
STAGING.mkdir(parents=True, exist_ok=True)

to_get = [
    ("annonces/annonces_immo.json", STAGING / "annonces_immo.json"),
    ("images/images_metadata.json", STAGING / "images_metadata.json"),
]

# geo: on prend ce qui existe dans MinIO
if "geographiques/donnees_geo.geojson" in objs:
    to_get.append(("geographiques/donnees_geo.geojson", STAGING / "donnees_geo.geojson"))
elif "geographiques/donnees_geo.csv" in objs:
    to_get.append(("geographiques/donnees_geo.csv", STAGING / "donnees_geo.csv"))
else:
    print("Aucun fichier geo trouvé dans MinIO (geojson/csv).")

for obj_name, dest in to_get:
    dest.parent.mkdir(parents=True, exist_ok=True)
    client.fget_object(BUCKET, obj_name, str(dest))
    print("Download OK:", obj_name, "->", dest)

Download OK: annonces/annonces_immo.json -> /Users/cefim/immobilier_pipeline/staging/annonces_immo.json
Download OK: images/images_metadata.json -> /Users/cefim/immobilier_pipeline/staging/images_metadata.json
Download OK: geographiques/donnees_geo.geojson -> /Users/cefim/immobilier_pipeline/staging/donnees_geo.geojson


In [13]:
import os

print("Contenu staging:", os.listdir(STAGING))
for f in os.listdir(STAGING):
    p = STAGING / f
    print(f, "-", p.stat().st_size, "bytes")

Contenu staging: ['donnees_geo.geojson', 'annonces_immo.json', 'images_metadata.json']
donnees_geo.geojson - 20565 bytes
annonces_immo.json - 32740 bytes
images_metadata.json - 66385 bytes


In [14]:
#ML + génération des catalogues (cellules à exécuter)

In [15]:
from pathlib import Path
from minio import Minio

# Chemins locaux
ROOT = Path.home() / "immobilier_pipeline"
STAGING = ROOT / "staging"
OUTPUTS = ROOT / "outputs"
MODELS = ROOT / "models"

OUTPUTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

# MinIO
MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
BUCKET = "immobilier-raw"

client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=False
)

print("OK - staging:", STAGING)
print("Fichiers staging:", [p.name for p in STAGING.iterdir()])
print("Buckets:", [b.name for b in client.list_buckets()])

OK - staging: /Users/cefim/immobilier_pipeline/staging
Fichiers staging: ['donnees_geo.geojson', 'annonces_immo.json', 'images_metadata.json']
Buckets: ['immobilier-curated', 'immobilier-raw', 'immobilier-refined']


In [16]:
#— ML prix estimé + flag fraude → catalog_biens.csv

In [17]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
import joblib

# Charger annonces
with open(STAGING / "annonces_immo.json", "r", encoding="utf-8") as f:
    annonces = json.load(f)
df = pd.DataFrame(annonces)

# Helper: trouver la 1ère colonne existante
def pick_col(possibles, default=None):
    for c in possibles:
        if c in df.columns:
            return c
    return default

col_bien_id = pick_col(["bien_id", "property_id", "id"])
col_type = pick_col(["type", "type_bien", "property_type"])
col_surface = pick_col(["surface", "surface_m2", "area"])
col_pieces = pick_col(["nb_pieces", "pieces", "rooms"])
col_ville = pick_col(["ville", "city"])
col_cp = pick_col(["code_postal", "cp", "postal_code"])
col_prix = pick_col(["prix", "price"])
col_annee = pick_col(["annee_construction", "year_built", "annee"])
col_equip = pick_col(["equipements", "amenities", "equipment"])

# Garantir un bien_id
if col_bien_id is None:
    df["bien_id"] = [f"BIEN_{i:04d}" for i in range(len(df))]
    col_bien_id = "bien_id"

# Normaliser dans les noms attendus du catalogue
out = pd.DataFrame()
out["bien_id"] = df[col_bien_id].astype(str)

out["type"] = df[col_type] if col_type else None
out["surface"] = pd.to_numeric(df[col_surface], errors="coerce") if col_surface else np.nan
out["nb_pieces"] = pd.to_numeric(df[col_pieces], errors="coerce") if col_pieces else np.nan
out["ville"] = df[col_ville] if col_ville else None
out["code_postal"] = df[col_cp] if col_cp else None
out["prix"] = pd.to_numeric(df[col_prix], errors="coerce") if col_prix else np.nan
out["annee_construction"] = pd.to_numeric(df[col_annee], errors="coerce") if col_annee else np.nan

# Equipements: garder en texte (si liste -> join)
if col_equip:
    def fmt_equip(x):
        if isinstance(x, list):
            return "|".join([str(v) for v in x])
        return None if pd.isna(x) else str(x)
    out["equipements"] = df[col_equip].apply(fmt_equip)
else:
    out["equipements"] = None

# Feature simple: nb d'équipements (utile ML)
out["nb_equipements"] = out["equipements"].fillna("").apply(lambda s: 0 if s == "" else len(s.split("|")))

# Pipeline ML (prix estimé)
num_features = ["surface", "nb_pieces", "nb_equipements"]
cat_features = ["type", "ville", "code_postal"]

X = out[num_features + cat_features].copy()
y = out["prix"].copy()

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_features),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_features),
    ]
)

model = Pipeline([
    ("prep", preprocess),
    ("rf", RandomForestRegressor(n_estimators=250, random_state=42))
])

mask = y.notna()
X_train, X_test, y_train, y_test = train_test_split(X[mask], y[mask], test_size=0.2, random_state=42)

model.fit(X_train, y_train)

out["prix_estime"] = model.predict(X).astype(float)

# Fraude (règle explicite simple et démontrable à l’examen)
out["ecart_relatif"] = (out["prix"] - out["prix_estime"]).abs() / (out["prix_estime"].abs() + 1e-6)
out["is_fraudulent"] = (out["ecart_relatif"] > 0.35).fillna(False)

# Sauvegarde modèle (pour montrer qu’on "fait du ML")
joblib.dump(model, MODELS / "modele_prix.joblib")

# Export catalogue biens (colonnes exactes demandées)
catalog_biens = out[[
    "bien_id","type","surface","nb_pieces","ville","code_postal",
    "prix","annee_construction","equipements","prix_estime","is_fraudulent"
]].copy()

path_biens = OUTPUTS / "catalog_biens.csv"
catalog_biens.to_csv(path_biens, index=False, sep=",")

print("OK - catalog_biens.csv ->", path_biens)
print(catalog_biens.head(5))

OK - catalog_biens.csv -> /Users/cefim/immobilier_pipeline/outputs/catalog_biens.csv
     bien_id         type  surface  nb_pieces     ville code_postal    prix  \
0  BIEN_0001  Appartement     70.0          3      Lyon        None  371790   
1  BIEN_0002           T4     68.9          4      Lyon        None  267386   
2  BIEN_0003       Maison     88.1          2      Lyon        None  429446   
3  BIEN_0004           T2     42.5          2  Bordeaux        None  157961   
4  BIEN_0005       Studio     19.6          1    Nantes        None   75218   

   annee_construction                           equipements  prix_estime  \
0                 NaN                               parking   268738.284   
1                 NaN  terrasse, ascenseur, parking, balcon   272331.040   
2                 NaN   balcon, ascenseur, terrasse, jardin   399519.892   
3                 NaN   ascenseur, jardin, balcon, terrasse   166250.572   
4                 NaN                      garage, terrasse 

In [18]:
#ML/score qualité images (sur métadonnées) → catalog_images_ml.csv

In [19]:
import json
import pandas as pd
import numpy as np

with open(STAGING / "images_metadata.json", "r", encoding="utf-8") as f:
    images = json.load(f)
img = pd.DataFrame(images)

def pick_img_col(possibles, default=None):
    for c in possibles:
        if c in img.columns:
            return c
    return default

col_image_id = pick_img_col(["image_id", "id"])
col_bien_id = pick_img_col(["bien_id", "property_id"])
col_type_piece = pick_img_col(["type_piece", "room_type", "piece"])
col_w = pick_img_col(["width", "w"])
col_h = pick_img_col(["height", "h"])
col_quality = pick_img_col(["quality_score", "score_qualite"])

# Garantir IDs
if col_image_id is None:
    img["image_id"] = [f"IMG_{i:05d}" for i in range(len(img))]
    col_image_id = "image_id"
if col_bien_id is None:
    img["bien_id"] = None
    col_bien_id = "bien_id"

catalog_images = pd.DataFrame()
catalog_images["image_id"] = img[col_image_id].astype(str)
catalog_images["bien_id"] = img[col_bien_id].astype(str)

catalog_images["type_piece"] = img[col_type_piece] if col_type_piece else None
catalog_images["width"] = pd.to_numeric(img[col_w], errors="coerce") if col_w else np.nan
catalog_images["height"] = pd.to_numeric(img[col_h], errors="coerce") if col_h else np.nan

# quality_score: si déjà présent -> on garde; sinon on calcule via résolution
if col_quality:
    catalog_images["quality_score"] = pd.to_numeric(img[col_quality], errors="coerce").clip(0, 1)
else:
    # Score simple démonstrable : résolution normalisée
    res = (catalog_images["width"].fillna(0) * catalog_images["height"].fillna(0))
    catalog_images["quality_score"] = (res / (1600*1200)).clip(0, 1)
    # Si pas de width/height, mettre un score moyen
    catalog_images.loc[res == 0, "quality_score"] = 0.6

catalog_images["is_low_quality"] = catalog_images["quality_score"] < 0.4
catalog_images["is_misleading"] = catalog_images["type_piece"].isna()  # règle simple

path_images = OUTPUTS / "catalog_images_ml.csv"
catalog_images.to_csv(path_images, index=False, sep=",")

print("OK - catalog_images_ml.csv ->", path_images)
print(catalog_images.head(5))

OK - catalog_images_ml.csv -> /Users/cefim/immobilier_pipeline/outputs/catalog_images_ml.csv
   image_id    bien_id type_piece  width  height  quality_score  \
0  IMG_0001  BIEN_0004    cuisine   4011    1223            1.0   
1  IMG_0002  BIEN_0090    cuisine   1144    1298            1.0   
2  IMG_0003  BIEN_0098  exterieur    441    1358            1.0   
3  IMG_0004  BIEN_0089        sdb   3045    1757            1.0   
4  IMG_0005  BIEN_0071    cuisine   3119    1491            1.0   

   is_low_quality  is_misleading  
0           False          False  
1           False          False  
2           False          False  
3           False          False  
4           False          False  


In [20]:
#Catalogue des fichiers (inventaire + hash) → catalog_files.csv

In [21]:
import hashlib, uuid
from datetime import datetime

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

rows = []
now = datetime.utcnow().isoformat()

# RAW (staging)
for p in sorted(STAGING.iterdir()):
    if p.is_file():
        rows.append({
            "file_id": str(uuid.uuid4()),
            "filename": p.name,
            "extension": p.suffix.replace(".", ""),
            "size_kb": round(p.stat().st_size / 1024, 2),
            "file_hash": sha256_file(p),
            "source_type": "raw",
        })

# CURATED (outputs)
for p in sorted(OUTPUTS.iterdir()):
    if p.is_file() and p.suffix.lower() == ".csv":
        rows.append({
            "file_id": str(uuid.uuid4()),
            "filename": p.name,
            "extension": p.suffix.replace(".", ""),
            "size_kb": round(p.stat().st_size / 1024, 2),
            "file_hash": sha256_file(p),
            "source_type": "curated",
        })

catalog_files = pd.DataFrame(rows)[["file_id","filename","extension","size_kb","file_hash","source_type"]]

path_files = OUTPUTS / "catalog_files.csv"
catalog_files.to_csv(path_files, index=False, sep=",")

print("OK - catalog_files.csv ->", path_files)
print(catalog_files)

OK - catalog_files.csv -> /Users/cefim/immobilier_pipeline/outputs/catalog_files.csv
                                file_id               filename extension  \
0  b183fcc6-1667-4ece-8e4c-06f444e8af2e     annonces_immo.json      json   
1  dd614971-5e47-4634-b451-1344afa9d040    donnees_geo.geojson   geojson   
2  3990e27a-45bc-4c8f-8d2f-d6bf2d98b470   images_metadata.json      json   
3  4f15881a-8af7-483e-b08f-40e6546b8c62      catalog_biens.csv       csv   
4  249adcb8-636e-4961-9d3f-c9bfba5697b6      catalog_files.csv       csv   
5  ef0bc9e1-d1df-48f9-8068-c2a06888545d  catalog_images_ml.csv       csv   

   size_kb                                          file_hash source_type  
0    31.97  656ef9f6ab86c1053fcea42680caa22aa9ee85756f7fd7...         raw  
1    20.08  26076559c3e892f98ebbce4b3e9288903fbf474a718b35...         raw  
2    64.83  0db7a3b6cf2b91efe9087e937abeab6d0b534497fa60c3...         raw  
3     8.87  c26310c29c783f67e653dcc152a1bca1b541294e37b2f1...     curated  
4 

In [22]:
#Upload des catalogues dans MinIO (zone curated)

In [23]:
# On stocke les catalogues dans le même bucket, sous un préfixe curated/
prefix = "curated/catalogues/"

for fname in ["catalog_files.csv", "catalog_images_ml.csv", "catalog_biens.csv"]:
    local_path = str(OUTPUTS / fname)
    obj_name = prefix + fname
    client.fput_object(BUCKET, obj_name, local_path)
    print("Upload OK:", obj_name)

print("Objets curated:")
for o in client.list_objects(BUCKET, prefix=prefix, recursive=True):
    print(" -", o.object_name, o.size)

Upload OK: curated/catalogues/catalog_files.csv
Upload OK: curated/catalogues/catalog_images_ml.csv
Upload OK: curated/catalogues/catalog_biens.csv
Objets curated:
 - curated/catalogues/catalog_biens.csv 9085
 - curated/catalogues/catalog_files.csv 886
 - curated/catalogues/catalog_images_ml.csv 10440


In [24]:
#Vérifier que les fichiers sont bien en local

In [25]:
from pathlib import Path

OUTPUTS = Path.home() / "immobilier_pipeline" / "outputs"
print("OUTPUTS =", OUTPUTS)
print(list(p.name for p in OUTPUTS.glob("*.csv")))

OUTPUTS = /Users/cefim/immobilier_pipeline/outputs
['catalog_images_ml.csv', 'catalog_files.csv', 'catalog_biens.csv']


In [26]:
#Uploader sur MinIO

In [27]:
from minio import Minio

client = Minio("localhost:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)
BUCKET = "immobilier-raw"
PREFIX = "curated/catalogues/"

for fname in ["catalog_biens.csv", "catalog_images_ml.csv", "catalog_files.csv"]:
    local_path = str(OUTPUTS / fname)
    client.fput_object(BUCKET, PREFIX + fname, local_path)
    print("✅ Upload:", PREFIX + fname)

✅ Upload: curated/catalogues/catalog_biens.csv
✅ Upload: curated/catalogues/catalog_images_ml.csv
✅ Upload: curated/catalogues/catalog_files.csv


In [28]:
for o in client.list_objects(BUCKET, prefix="curated/catalogues/", recursive=True):
    print(o.object_name, o.size)

curated/catalogues/catalog_biens.csv 9085
curated/catalogues/catalog_files.csv 886
curated/catalogues/catalog_images_ml.csv 10440


In [57]:
print("kernel OK")

kernel OK


In [59]:
from minio import Minio
from pathlib import Path

# 1) Connexion MinIO (si tu es en http local, secure=False)
client = Minio(
    "localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)

BUCKET = "immobilier-raw"
PREFIX = "curated/catalogues/"

# 2) Dossier local de sortie
out_dir = Path.cwd() / "curated_from_minio"
out_dir.mkdir(parents=True, exist_ok=True)

# 3) Télécharger les fichiers curatés
files = ["catalog_biens.csv", "catalog_images_ml.csv", "catalog_files.csv"]
for fname in files:
    client.fget_object(BUCKET, PREFIX + fname, str(out_dir / fname))
    print("Téléchargé:", out_dir / fname)

Téléchargé: /Users/cefim/curated_from_minio/catalog_biens.csv
Téléchargé: /Users/cefim/curated_from_minio/catalog_images_ml.csv
Téléchargé: /Users/cefim/curated_from_minio/catalog_files.csv


In [61]:
import os
print(os.listdir("curated_from_minio"))

['catalog_images_ml.csv', 'catalog_files.csv', 'catalog_biens.csv']


In [ ]:
#nettoyage 

In [67]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("/Users/cefim/curated_from_minio")

print("BASE_DIR existe ?", BASE_DIR.exists(), "|", BASE_DIR)
print("CSV présents :", [p.name for p in BASE_DIR.glob("*.csv")])

def clean_csv(path_in, path_out, id_col):
    path_in = Path(path_in)
    path_out = Path(path_out)

    if not path_in.exists():
        raise FileNotFoundError(f"❌ Fichier introuvable: {path_in}")

    # Lecture
    df = pd.read_csv(path_in, sep=",", low_memory=False)

    # Nettoyage des noms de colonnes (très important)
    df.columns = df.columns.astype(str).str.strip()

    if id_col not in df.columns:
        raise KeyError(f"❌ Colonne '{id_col}' introuvable dans {path_in.name}. Colonnes: {list(df.columns)}")

    # 1) Supprimer les lignes d'en-tête répétées (ex: 'bien_id' dans la colonne bien_id)
    df = df[df[id_col].astype(str).str.strip().ne(id_col)]

    # 2) Remplacer certains "nan" texte / chaînes vides par NA
    df = df.replace(to_replace=r"^\s*(nan|NaN|None|null)\s*$", value=np.nan, regex=True)

    # 3) Trim sur toutes les colonnes texte
    obj_cols = df.select_dtypes(include=["object"]).columns
    for c in obj_cols:
        df[c] = (
            df[c]
            .astype("string")
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "null": pd.NA})
        )

    # Écriture
    df.to_csv(path_out, index=False, sep=",")
    print(f"✅ OK: {path_out.name} | rows={len(df)} | cols={len(df.columns)}")

# Fichiers attendus + clé ID
tasks = [
    ("catalog_biens.csv", "catalog_biens_clean.csv", "bien_id"),
    ("catalog_images_ml.csv", "catalog_images_ml_clean.csv", "image_id"),
    ("catalog_files.csv", "catalog_files_clean.csv", "file_id"),
]

for fin, fout, idcol in tasks:
    clean_csv(BASE_DIR / fin, BASE_DIR / fout, idcol)

BASE_DIR existe ? True | /Users/cefim/curated_from_minio
CSV présents : ['catalog_images_ml.csv', 'catalog_files.csv', 'catalog_biens.csv']
✅ OK: catalog_biens_clean.csv | rows=120 | cols=11
✅ OK: catalog_images_ml_clean.csv | rows=200 | cols=8
✅ OK: catalog_files_clean.csv | rows=6 | cols=6


/var/folders/55/01s82b4x5fb6mfhbkx3gnvd40000gp/T/ipykernel_15977/163843810.py:33: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include=["object"]).columns
/var/folders/55/01s82b4x5fb6mfhbkx3gnvd40000gp/T/ipykernel_15977/163843810.py:33: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pan

In [ ]:
#vérification 

In [69]:
from pathlib import Path
BASE_DIR = Path("/Users/cefim/curated_from_minio")

for p in sorted(BASE_DIR.glob("*clean.csv")):
    print("✅", p.name, "-", p.stat().st_size, "bytes")

✅ catalog_biens_clean.csv - 9085 bytes
✅ catalog_files_clean.csv - 886 bytes
✅ catalog_images_ml_clean.csv - 10440 bytes


In [73]:
from pathlib import Path
geo_path = Path("/Users/cefim/donnees_geo.json")
print("donnees_geo.json existe ?", geo_path.exists(), "|", geo_path)

donnees_geo.json existe ? True | /Users/cefim/donnees_geo.json


In [75]:
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/cefim/curated_from_minio")
geo_json = Path("/Users/cefim/donnees_geo.json")  # adapte si besoin

# 1) Lire GeoJSON -> geo_points.csv
with open(geo_json, "r", encoding="utf-8") as f:
    geo = json.load(f)

rows = []
for feat in geo["features"]:
    lon, lat = feat["geometry"]["coordinates"]
    p = feat["properties"]
    rows.append({
        "geo_id": p["id"],
        "ville": p["ville"],
        "lon": lon,
        "lat": lat,
        "proximite_commerce": p["proximite_commerce"],
        "proximite_ecole": p["proximite_ecole"],
        "proximite_transport": p["proximite_transport"],
        "score_proximite_global": p["score_proximite_global"],
    })

geo_points = pd.DataFrame(rows)
geo_points.to_csv(BASE_DIR / "geo_points.csv", index=False, sep=",")
print("✅ geo_points.csv =", len(geo_points))

# 2) Créer bien_geo.csv (association bien -> geo par ville en round-robin)
biens = pd.read_csv(BASE_DIR / "catalog_biens_clean.csv", sep=",")
mapping = []

for ville, group_biens in biens.groupby("ville"):
    geo_ville = geo_points[geo_points["ville"] == ville]
    if len(geo_ville) == 0:
        continue
    geo_ids = geo_ville["geo_id"].tolist()
    for i, bien_id in enumerate(group_biens["bien_id"].tolist()):
        mapping.append({"bien_id": bien_id, "geo_id": geo_ids[i % len(geo_ids)]})

bien_geo = pd.DataFrame(mapping)
bien_geo.to_csv(BASE_DIR / "bien_geo.csv", index=False, sep=",")
print("✅ bien_geo.csv =", len(bien_geo))

✅ geo_points.csv = 50
✅ bien_geo.csv = 120


In [77]:
from pathlib import Path
BASE_DIR = Path("/Users/cefim/curated_from_minio")
sorted([p.name for p in BASE_DIR.glob("*.csv")])

['bien_geo.csv',
 'catalog_biens.csv',
 'catalog_biens_clean.csv',
 'catalog_files.csv',
 'catalog_files_clean.csv',
 'catalog_images_ml.csv',
 'catalog_images_ml_clean.csv',
 'geo_points.csv']